In [14]:
import pandas as pd


In [15]:
# INPUTS
SEASON = '2024-2025'
LEAGUE_SLUG = 'Premier-League'

In [16]:
expected_pace_path = f'../../data/features/context/expected_pace_{SEASON}_{LEAGUE_SLUG}.parquet'

exp = pd.read_parquet(expected_pace_path)


In [17]:
fixtures = pd.read_csv(f'../../data/raw/fbref/fixtures_{SEASON}_{LEAGUE_SLUG}.csv')
team_stats = pd.read_csv(f'../../data/raw/fbref/team_stats_{SEASON}_{LEAGUE_SLUG}.csv')

In [18]:
team_stats[['match_id','home_shots','away_shots']]

,match_id,home_shots,away_shots
0,cc5b4244,14,10
1,a1d0d529,7,18
2,34557647,3,19
3,71618ace,9,10
4,4efc72e4,14,13
...,...,...,...
375,464cbad6,10,6
376,3d22336e,13,19
377,36844e73,17,14
378,1ff370e8,20,3


In [24]:
exp

,match_id,exp_pace_home,exp_pace_away,exp_pace_total
0,cc5b4244,NaN,NaN,NaN
1,a1d0d529,NaN,NaN,NaN
2,34557647,NaN,NaN,NaN
3,71618ace,NaN,NaN,NaN
4,4efc72e4,NaN,NaN,NaN
...,...,...,...,...
375,464cbad6,10.320174,16.132674,26.452847
376,3d22336e,9.762024,13.012024,22.774048
377,36844e73,13.112573,8.800073,21.912646
378,1ff370e8,14.889383,9.889383,24.778766


In [19]:
exp['exp_pace_home'].describe()

count    370.000000
mean      12.842886
std        2.134654
min        7.346709
25%       11.428442
50%       12.653009
75%       14.372347
max       19.161560
Name: exp_pace_home, dtype: float64

In [20]:
exp

,match_id,exp_pace_home,exp_pace_away,exp_pace_total
0,cc5b4244,NaN,NaN,NaN
1,a1d0d529,NaN,NaN,NaN
2,34557647,NaN,NaN,NaN
3,71618ace,NaN,NaN,NaN
4,4efc72e4,NaN,NaN,NaN
...,...,...,...,...
375,464cbad6,10.320174,16.132674,26.452847
376,3d22336e,9.762024,13.012024,22.774048
377,36844e73,13.112573,8.800073,21.912646
378,1ff370e8,14.889383,9.889383,24.778766


In [21]:
df = fixtures.merge(team_stats[["match_id","home_shots","away_shots"]], on="match_id", how="inner")
df = df.merge(exp[["match_id","exp_pace_home","exp_pace_away","exp_pace_total"]], on="match_id", how="inner")
df["shots_total"] = df["home_shots"] + df["away_shots"]

df[~df['exp_pace_total'].isna()][["exp_pace_total","shots_total"]].corr()           # correlation
# (df["shots_total"] - df["exp_pace_total"]).describe() # residuals


,exp_pace_total,shots_total
exp_pace_total,1.000000,0.194036
shots_total,0.194036,1.000000


In [22]:
test = df[~df['exp_pace_total'].isna()][["exp_pace_total","shots_total"]]

test['diff'] = test['exp_pace_total'] - test['shots_total']
test['diff'].describe()



count    370.000000
mean      -0.071115
std        6.073846
min      -20.515246
25%       -4.220106
50%        0.400683
75%        3.914432
max       13.475237
Name: diff, dtype: float64

In [23]:
bins = pd.qcut(df["exp_pace_total"], 10, duplicates="drop")
df.groupby(bins)["shots_total"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pace_total"].mean())


/var/folders/l_/pcb32qsn4vz5n55rmw_4dctc0000gn/T/ipykernel_68251/2236549888.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(bins)["shots_total"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pace_total"].mean())


,obs,pred
exp_pace_total,,
"(19.052, 22.357]",23.297297,21.521978
"(22.357, 23.526]",24.108108,22.950649
"(23.526, 24.295]",25.837838,23.952051
"(24.295, 24.9]",25.513514,24.640313
"(24.9, 25.629]",25.027027,25.298949
"(25.629, 26.314]",25.783784,25.939346
"(26.314, 27.091]",26.648649,26.667470
"(27.091, 27.848]",26.675676,27.495974
"(27.848, 28.691]",26.810811,28.253114
